# Limpieza de matriculados
Se consolidan las tres materias en `data/matriculados/matriculados.parquet`. Se omite el título del Excel y se excluyen las filas de conteo y las filas vacías. Solo se completa `instancia` con `Libre` cuando está vacía y `estado_insc` es `Baja`.

También se normalizan encabezados y espacios, se convierten fechas e identificadores numéricos y se agrega la materia y el archivo de origen. Por cada alumno (identificación) y materia se conserva únicamente el registro con el mayor número de transacción, cualquiera sea su estado. Los controles detienen la exportación si quedan instancias nulas o datos obligatorios inválidos.

In [ ]:
from pathlib import Path
import pandas as pd

# Permite ejecutar desde la raíz del proyecto o desde jupyter/.
raiz = Path.cwd() if (Path.cwd() / 'data/matriculados').is_dir() else Path.cwd().parent
carpeta = raiz / 'data/matriculados'
archivos = sorted(a for a in carpeta.glob('*.xlsx') if not a.name.startswith('~$'))
if len(archivos) != 3:
    raise ValueError(f'Se esperaban 3 archivos de materias; se encontraron {len(archivos)}.')

def a_snake_case(columnas):
    return (columnas.str.strip().str.lower()
            .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')
            .str.replace(r'\W+', '_', regex=True).str.strip('_'))


In [ ]:
dfs = []
resumen = []
columnas = ['alumno', 'identificacion', 'comision', 'estado_insc', 'instancia',
            'fecha_inscripcion', 'nro_de_transaccion']

for archivo in archivos:
    df = pd.read_excel(archivo, header=1)
    df.columns = a_snake_case(df.columns)
    df = df.dropna(axis=1, how='all')
    if set(df.columns) != set(columnas):
        raise ValueError(f'{archivo.name}: encabezados inesperados: {df.columns.tolist()}')
    df = df[columnas].copy()

    # Los blancos y las cadenas de espacios se consideran nulos.
    for columna in df.select_dtypes(include=['object', 'string']).columns:
        df[columna] = df[columna].map(lambda valor: valor.strip() if isinstance(valor, str) else valor)
        df[columna] = df[columna].replace(r'^\s*$', pd.NA, regex=True)

    es_conteo = df['alumno'].astype('string').str.fullmatch(
        r'(?i)cantidad\s+de\s+filas\s*:?\s*', na=False)
    es_vacia = df.isna().all(axis=1)
    df = df.loc[~(es_conteo | es_vacia)].copy()

    completar = df['instancia'].isna() & df['estado_insc'].astype('string').str.casefold().eq('baja').fillna(False)
    df.loc[completar, 'instancia'] = 'Libre'

    # errors='raise' evita ocultar valores inválidos mediante conversiones a nulo.
    df['fecha_inscripcion'] = pd.to_datetime(df['fecha_inscripcion'], format='%d/%m/%Y %H:%M:%S', errors='raise')
    for columna in ['comision', 'nro_de_transaccion']:
        df[columna] = pd.to_numeric(df[columna], errors='raise').astype('Int64')
    for columna in ['alumno', 'identificacion', 'estado_insc', 'instancia']:
        df[columna] = df[columna].astype('string')

    nulos = df[columnas].isna().sum()
    if nulos.any():
        raise ValueError(f'{archivo.name}: quedan nulos que requieren revisión: {nulos[nulos > 0].to_dict()}')

    df['materia'] = archivo.stem
    df['archivo'] = archivo.name
    resumen.append({
        'materia': archivo.stem, 'filas': len(df),
        'filas_conteo_excluidas': int(es_conteo.sum()),
        'filas_vacias_excluidas': int(es_vacia.sum()),
        'instancias_completadas': int(completar.sum()),
        'filas_duplicadas': int(df.duplicated().sum()),
        'identificaciones_repetidas': int(df['identificacion'].duplicated().sum()),
        'transacciones_repetidas': int(df['nro_de_transaccion'].duplicated().sum()),
    })
    dfs.append(df)

df_matriculados = pd.concat(dfs, ignore_index=True)[['archivo', 'materia', *columnas]]
# La última inscripción se determina por transacción, dentro de cada alumno y materia.
filas_antes = len(df_matriculados)
df_matriculados = (df_matriculados
                   .sort_values('nro_de_transaccion', kind='stable')
                   .drop_duplicates(subset=['materia', 'identificacion'], keep='last')
                   .sort_index().reset_index(drop=True))
print(f'Inscripciones anteriores excluidas: {filas_antes - len(df_matriculados)}')
resumen_limpieza = pd.DataFrame(resumen)
resumen_limpieza['filas_exportadas'] = resumen_limpieza['materia'].map(df_matriculados.groupby('materia').size())
display(resumen_limpieza)
display(df_matriculados.isna().sum().rename('nulos'))


In [ ]:
salida = carpeta / 'matriculados.parquet'
df_matriculados.to_parquet(salida, index=False)
# Verifica que la lectura conserve filas, valores y tipos.
pd.testing.assert_frame_equal(df_matriculados, pd.read_parquet(salida))
print(f'Exportadas {len(df_matriculados)} inscripciones a {salida}')
display(df_matriculados.head())
